# S25 · Power and sample size - how many users do we need?

We see the two ways a test can be wrong as two overlapping bell curves, measure our
chance of catching a real effect (its **power**) two different ways, and then answer
the question every product manager asks: how many users do we need before we launch?

**New here? Read this once.**

- New to Python? Press play on each cell, top to bottom, and read the note above each
  one. The pictures do most of the teaching here.
- Fuzzy on averages, spread and bell curves? Open
  `primers/what_is_a_distribution.md`.
- Already confident? Look for the cell marked **Stretch (optional)** near the end.
- Any unfamiliar word is in `primers/glossary.md`.

## Setup

On **Google Colab**, run the next cell once. On your **own machine** you already
installed everything with `uv`, so it does nothing there.

In [ ]:
# This notebook uses numpy, matplotlib, scipy and statsmodels.
# Google Colab already ships all four, so there is nothing to install.
print("Setup complete - nothing to install.")

## Two ways to be wrong

When we make a yes/no decision under uncertainty, two mistakes are possible:

- **Type I error (false positive):** we declare B better when it truly is not. Its
  probability is called **alpha**, usually 0.05.
- **Type II error (false negative):** we miss a real improvement in B. Its probability
  is called **beta**.

**Statistical power = 1 - beta** is the chance we DO catch a real effect. The
conventional target is 80% power. This notebook makes all of that concrete, and then
shows how a bigger sample buys more power.

## Step 1 - import our tools

`scipy.stats` to draw the bell curves; a ready-made power calculator from
`statsmodels`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# statsmodels has ready-made power calculators. This one is for comparing two
# averages with a t-test.
from statsmodels.stats.power import TTestIndPower

# Set a seed so we all get the same random numbers, and the same answers, every time.
np.random.seed(0)

print("Tools ready.")

## Step 2 - picture the two error types

Imagine two worlds, each drawn as a bell curve of the test statistic:

- The **left** bell (H0): B does nothing.
- The **right** bell (H1): B really helps.

We pick a decision threshold. The piece of the H0 curve to the RIGHT of the line is the
**Type I** error (we wrongly declare a win). The piece of the H1 curve to the LEFT of
the line is the **Type II** error (we wrongly miss a real win).

In [ ]:
# A grid of test-statistic values to draw the curves over.
x = np.linspace(-4, 8, 500)

# H0: centred at 0 (no effect). H1: centred at 3.2 (a real effect).
curve_H0 = stats.norm.pdf(x, 0, 1)
curve_H1 = stats.norm.pdf(x, 3.2, 1)

# Our decision threshold: reject H0 if the statistic lands to the right of this.
threshold = 1.9

plt.figure(figsize=(9, 5))
plt.plot(x, curve_H0, color="#2E75B6", label="H0: no effect")
plt.plot(x, curve_H1, color="#27AE60", label="H1: real effect")

# Shade the Type I error: part of H0 to the RIGHT of the threshold.
right_of_line = x >= threshold
plt.fill_between(x[right_of_line], curve_H0[right_of_line],
                 color="#C0392B", alpha=0.5, label="Type I error (alpha)")

# Shade the Type II error: part of H1 to the LEFT of the threshold.
left_of_line = x <= threshold
plt.fill_between(x[left_of_line], curve_H1[left_of_line],
                 color="#E59866", alpha=0.6, label="Type II error (beta)")

plt.axvline(threshold, color="grey", linestyle="--")
plt.xlabel("test statistic")
plt.ylabel("density")
plt.title("Two error types: alpha (false positive) and beta (false negative)")
plt.legend()
plt.show()

### Read the picture

Move the dashed line in your head. Slide it RIGHT and the red area (Type I) shrinks but
the orange area (Type II) grows. You cannot drive both to zero at once with the
threshold alone. The way to shrink BOTH is to push the two bells apart, and a bigger
sample is exactly what does that.

## Step 3 - measure power by simulation (to build intuition)

Before using any formula, let us just *count*. We run many pretend A/B tests where B
truly is better, and see what fraction of them we correctly flag as significant. That
fraction IS the power.

In [ ]:
# The true situation: B's average is a bit higher than A's.
true_mean_A = 11.0
true_mean_B = 11.8     # B is genuinely better by 0.8
spread = 2.6           # the spread (standard deviation) in each group
sample_size_per_group = 120

number_of_experiments = 2000
times_we_detected_the_effect = 0

# Run many independent pretend experiments.
for experiment in range(number_of_experiments):
    # Draw a fresh sample for each group.
    sample_A = np.random.normal(true_mean_A, spread, size=sample_size_per_group)
    sample_B = np.random.normal(true_mean_B, spread, size=sample_size_per_group)

    # Run the t-test on this experiment.
    t_statistic, p_value = stats.ttest_ind(sample_A, sample_B, equal_var=False)

    # Did we correctly call it significant?
    if p_value < 0.05:
        times_we_detected_the_effect = times_we_detected_the_effect + 1

# Power is the fraction of experiments where we caught the real effect.
estimated_power = times_we_detected_the_effect / number_of_experiments
print("Out of", number_of_experiments, "experiments, we detected the effect",
      times_we_detected_the_effect, "times.")
print("Estimated power by simulation:", round(estimated_power, 3))

## Step 4 - how many users do we need?

This is the headline of the whole notebook, and the one number to take away: the
question every product manager asks before a launch. Fix the improvement we care
about, the alpha, and the power we want (80%), and `statsmodels` solves for the
required sample size in a single call.

In [ ]:
# statsmodels has a ready-made power / sample-size calculator for comparing two
# averages with a t-test.
power_calculator = TTestIndPower()

# The calculator needs the improvement scaled by the natural spread in each group.
# That single scaled number is all it needs here; the Stretch cell below unpacks
# exactly what it means.
effect_size = (true_mean_B - true_mean_A) / spread

# The practical question: for 80% power at alpha = 0.05, how big must each group be?
required_sample_size = power_calculator.solve_power(
    effect_size=effect_size,
    power=0.80,
    alpha=0.05,
    alternative="two-sided")

# Sample size must be a whole number, and we round UP to be safe.
required_sample_size = int(np.ceil(required_sample_size))

print("How many users do we need before we launch?")
print("For 80% power we need about", required_sample_size, "users PER GROUP,")
print("so roughly", 2 * required_sample_size, "users in total across A and B.")

## Step 5 - plot power against sample size

Power climbs as the sample grows, because a larger sample shrinks the noise and pulls
the two bells apart. We mark the 80% target line and the sample size we just
computed.

In [ ]:
# A range of sample sizes per group to try.
sample_sizes = np.arange(10, 600, 10)

# Compute the power at each sample size, one at a time, into a list.
powers = []
for n in sample_sizes:
    p = power_calculator.power(
        effect_size=effect_size,
        nobs1=n,
        alpha=0.05,
        alternative="two-sided")
    powers.append(p)

plt.figure(figsize=(8, 5))
plt.plot(sample_sizes, powers, color="#2E75B6", linewidth=2.5)

# The conventional 80% power target.
plt.axhline(0.80, color="#27AE60", linestyle="--", label="80% target")

# Mark the sample size we computed in Step 4.
plt.axvline(required_sample_size, color="grey", linestyle=":",
            label="needed n = " + str(required_sample_size))

plt.xlabel("sample size per group")
plt.ylabel("power  (1 - beta)")
plt.title("Power rises with sample size")
plt.ylim(0, 1.02)
plt.legend()
plt.show()

### Stretch (optional) - the effect size behind the number (Cohen's d)

Skip this if you are new to statistics. The power calculator above did not use the
averages and the spread separately - it needed a single **standardised effect size**,
called Cohen's d = difference in averages divided by the spread. It measures the gap
between A and B in units of the natural spread (standard deviations), so the same d
gives the same power whatever the raw numbers happen to be. Below we print d, and check
that the formula's power matches the simulation from Step 3.

In [ ]:
# Cohen's d: how big the effect is, measured in standard deviations.
print("effect size (Cohen's d):", round(effect_size, 3))

# The formula's power at the sample size we simulated with in Step 3.
power_from_formula = power_calculator.power(
    effect_size=effect_size,
    nobs1=sample_size_per_group,   # sample size per group
    alpha=0.05,
    alternative="two-sided")

print("Power from the formula  :", round(power_from_formula, 3))
print("Power from simulation   :", round(estimated_power, 3))
print("They agree closely - the simulation confirms the formula.")

### Stretch (optional) - smaller effects cost a lot more users

Skip this if you are new to statistics. Here is a fact that surprises people and
matters for planning: to detect an effect half as large, you need far more than twice
the users. Let us halve the effect size and ask for the sample size again.

In [ ]:
# Half the effect we used before.
smaller_effect = effect_size / 2

users_for_smaller_effect = power_calculator.solve_power(
    effect_size=smaller_effect,
    power=0.80,
    alpha=0.05,
    alternative="two-sided")
users_for_smaller_effect = int(np.ceil(users_for_smaller_effect))

print("effect size", round(effect_size, 3), "needs about", required_sample_size, "per group")
print("effect size", round(smaller_effect, 3), "needs about", users_for_smaller_effect, "per group")
print()
print("Halving the effect roughly QUADRUPLES the users needed.")
print("That is why chasing tiny lifts can be very expensive to prove.")

## What you just did

You answered the practical question at the heart of today's math object:

- computed the **sample size** needed for 80% power - the headline number to take away,
- saw **Type I** and **Type II** errors as two overlapping bell curves, and
- plotted **power versus sample size**.

(The Stretch cells unpack the effect size, Cohen's d, that sits behind that number, and
confirm the formula against a simulation.)

The practical lesson: decide your sample size BEFORE you launch. An underpowered test
that finds 'no effect' has told you almost nothing, because you cannot tell a true
'nothing here' from a real effect you were simply too small to see.